In [1]:
import os
import warnings
import boto3
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

#### Functions

In [2]:
# feature engineer
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom sklearn transformer for credit risk feature engineering.
    - Engineers int_age from dob: age is a known predictor of creditworthiness.
      Stores a reference date during fit() so that age calculation is
      reproducible and consistent between training and serving, avoiding
      train-serve skew over time.
    - Engineers flt_payment_to_income: payment-to-income ratio is a standard
      credit risk feature that measures borrower capacity.
    Note: no columns are dropped here. Column exclusion is handled in 04_model
    to keep preprocessing fast.
    """
    def fit(self, X, y=None):
        self.reference_date_ = pd.Timestamp.now()
        return self

    def transform(self, X):
        df = X.copy()
        # engineer age from dob
        if 'dob' in df.columns:
            df['dob'] = pd.to_datetime(df['dob'])
            df['int_age'] = ((self.reference_date_ - df['dob']).dt.days / 365.25).astype(int)
        # engineer payment to income ratio
        if 'loan_amount' in df.columns and 'stated_income' in df.columns:
            df['flt_payment_to_income'] = df['loan_amount'] / df['stated_income'].replace(0, np.nan)
        return df

In [3]:
# text standardizer
class TextStandardizer(BaseEstimator, TransformerMixin):
    """
    Custom sklearn transformer that lowercases and strips whitespace
    from all string values. This prevents duplicate categories caused
    by inconsistent casing (e.g., "Mobile" vs "mobile").
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        import numpy as np
        arr = np.array(X, dtype=object)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                if isinstance(arr[i, j], str):
                    arr[i, j] = arr[i, j].lower().strip()
        return arr

In [4]:
# plot missing before and after preprocessing
def plot_missing_before_after(df_before, df_after, str_filename='output/missing_before_after.png'):
    ser_before = df_before.isnull().mean()
    ser_after = df_after.isnull().mean()
    # only show columns that had missing values before
    ser_before = ser_before[ser_before > 0].sort_values(ascending=True)
    if len(ser_before) == 0:
        print('No missing values found before preprocessing.')
        return
    ser_after = ser_after.reindex(ser_before.index).fillna(0)
    # plot
    list_cols = ser_before.index.tolist()
    arr_y = np.arange(len(list_cols))
    flt_bar_height = 0.35
    fig, ax = plt.subplots(figsize=(10, max(5, len(list_cols) * 0.5)))
    ax.barh(arr_y + flt_bar_height / 2, ser_before.values, flt_bar_height, label='Before', color='salmon', edgecolor='black')
    ax.barh(arr_y - flt_bar_height / 2, ser_after.values, flt_bar_height, label='After', color='steelblue', edgecolor='black')
    ax.set_yticks(arr_y)
    ax.set_yticklabels(list_cols)
    ax.set_title('Proportion Missing Before and After Preprocessing', fontsize=16)
    ax.set_xlabel('Proportion Missing', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [5]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 input path
str_s3_input = f's3://{str_bucket}/02_data_split'
print(f'S3 Input: {str_s3_input}')

# target
str_target = 'default_12m'
print(f'Target: {str_target}')

# output directory
os.makedirs('output', exist_ok=True)

Bucket: credit-risk-claude
Step: 03_preprocessing
S3 Input: s3://credit-risk-claude/02_data_split
Target: default_12m


#### Read Data

In [6]:
# read data
df_train = pd.read_parquet(f'{str_s3_input}/df_train.parquet')
df_valid = pd.read_parquet(f'{str_s3_input}/df_valid.parquet')
df_test = pd.read_parquet(f'{str_s3_input}/df_test.parquet')

# print shapes
for str_name, df_split in [('Train', df_train), ('Validation', df_valid), ('Test', df_test)]:
    print(f'{str_name}: {df_split.shape}')

Train: (17715, 20)
Validation: (3796, 20)
Test: (3797, 20)


#### Data Contract Validation

Validate that incoming data matches the expected schema before processing. This catches upstream schema changes, missing columns, and data quality issues early.

In [7]:
# data contract validation
list_expected_cols = ['loan_id', 'origination_date', 'dob', 'loan_amount', 'term_months',
                      'channel', 'employment_length_years', 'stated_income', 'state',
                      'has_prior_loans_with_us', 'bureau_score', 'open_trades', 'delinq_12m',
                      'utilization', 'inquiries_6m', 'public_records', 'charged_off_amount',
                      'paid_interest_amount', 'apr', str_target]

for str_name, df_split in [('Train', df_train), ('Validation', df_valid), ('Test', df_test)]:
    # check expected columns exist
    list_missing = [col for col in list_expected_cols if col not in df_split.columns]
    assert len(list_missing) == 0, f'{str_name} missing columns: {list_missing}'
    # check no empty splits
    assert len(df_split) > 0, f'{str_name} has no rows'
    # check target is binary
    assert set(df_split[str_target].dropna().unique()).issubset({0, 1}), f'{str_name} target is not binary'

print('Data contract validation passed')

Data contract validation passed


#### Feature Engineering

The first step derives new features and identifies column types. The `FeatureEngineer` transformer creates age from date of birth and a payment-to-income ratio. We run it here to determine which columns are numeric vs. categorical for the next pipeline step. Column dropping (leaky variables, IDs, etc.) is deferred to the modeling notebook where feature selection decisions are made.

In [8]:
# apply feature engineering to determine column names
feature_engineer = FeatureEngineer()
df_train_fe = feature_engineer.transform(df_train)

# separate target
list_feature_cols = [col for col in df_train_fe.columns if col != str_target]

# identify numeric and categorical columns
list_numeric_cols = df_train_fe[list_feature_cols].select_dtypes(include=[np.number]).columns.tolist()
list_categorical_cols = df_train_fe[list_feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features ({len(list_numeric_cols)}): {list_numeric_cols}')
print(f'Categorical features ({len(list_categorical_cols)}): {list_categorical_cols}')

AttributeError: 'FeatureEngineer' object has no attribute 'reference_date_'

#### Numeric Preprocessing

Numeric features are imputed using the **median**. Median is preferred over mean because credit risk data often has skewed distributions (e.g., income, loan amounts), and the median is robust to outliers. No scaling is applied because XGBoost is a tree-based algorithm that splits on thresholds, making it invariant to monotonic transformations of the features.

In [ ]:
# numeric transformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

#### Categorical Preprocessing

Categorical features are first standardized by lowercasing and stripping whitespace to prevent duplicate categories caused by inconsistent casing (e.g., "Mobile" vs "mobile"). Then missing values are imputed with the constant string "missing", treating missingness as its own category. This is appropriate for credit risk because the absence of data can itself be informative (e.g., a missing employment length may indicate informal employment). Then **ordinal encoding** is applied. Ordinal encoding is preferred over one-hot encoding for XGBoost because it avoids creating high-dimensional sparse features and XGBoost can learn effective splits on ordinal values. The integer assignments are purely alphabetical and do not consider the target variable, avoiding the leakage risk associated with target encoding. Unknown categories encountered at inference time are encoded as -1.

In [ ]:
# categorical transformer
categorical_transformer = Pipeline(steps=[
    ('standardizer', TextStandardizer()),
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

#### Assemble Pipeline

The full pipeline chains the `FeatureEngineer` with a `ColumnTransformer` that routes numeric and categorical features to their respective preprocessing steps. This ensures the entire preprocessing workflow is encapsulated in a single serializable object that can be saved and reused in the API for inference.

In [ ]:
# assemble full pipeline
column_transformer = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, list_numeric_cols),
        ('cat', categorical_transformer, list_categorical_cols),
    ],
)

pipeline = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()),
    ('column_transformer', column_transformer),
])

print('Pipeline defined successfully')
print(pipeline)

#### Fit and Transform

The pipeline is fit on the training data only. This is critical to prevent data leakage. If we fit on validation or test data, the imputed medians and encoder mappings would contain information from data the model should never see during training. After fitting on train, we transform all three splits using the same learned parameters.

#### Fit

In [ ]:
warnings.filterwarnings('ignore')

# fit on training data ONLY
pipeline.fit(df_train)

#### Save

In [ ]:
# save pipeline immediately after fitting
joblib.dump(pipeline, 'output/preprocessing_pipeline.joblib')
print('Pipeline saved to output/preprocessing_pipeline.joblib')

#### Learned Parameters

Inspecting the fitted pipeline's learned parameters increases transparency and auditability. This allows stakeholders to answer questions like "what value was imputed for bureau_score?" without needing to load the serialized pipeline object.

In [ ]:
# numeric imputer: learned median values
numeric_imputer = pipeline.named_steps['column_transformer'].named_transformers_['num'].named_steps['imputer']
df_medians = pd.DataFrame({
    'feature': list_numeric_cols,
    'median': numeric_imputer.statistics_,
}).sort_values('feature').reset_index(drop=True)
print('Numeric Imputation (median):')
display(df_medians)

# categorical encoder: learned category mappings
cat_pipeline = pipeline.named_steps['column_transformer'].named_transformers_['cat']
encoder = cat_pipeline.named_steps['encoder']
print('\nCategorical Encoding (ordinal):')
for str_col, arr_categories in zip(list_categorical_cols, encoder.categories_):
    df_mapping = pd.DataFrame({
        'category': arr_categories,
        'encoded_value': range(len(arr_categories)),
    })
    print(f'\n{str_col}:')
    display(df_mapping)

#### Transform

In [ ]:
# transform all splits
list_col_names = list_numeric_cols + list_categorical_cols

arr_train = pipeline.transform(df_train)
arr_valid = pipeline.transform(df_valid)
arr_test = pipeline.transform(df_test)

# reconstruct DataFrames
df_train_clean = pd.DataFrame(arr_train, columns=list_col_names)
df_train_clean[str_target] = df_train[str_target].values

df_valid_clean = pd.DataFrame(arr_valid, columns=list_col_names)
df_valid_clean[str_target] = df_valid[str_target].values

df_test_clean = pd.DataFrame(arr_test, columns=list_col_names)
df_test_clean[str_target] = df_test[str_target].values

warnings.filterwarnings('default')

# print shapes
for str_name, df_split in [('Train', df_train_clean), ('Validation', df_valid_clean), ('Test', df_test_clean)]:
    print(f'{str_name}: {df_split.shape}')

df_train_clean.head()

#### Missing Values Before and After

In [ ]:
plot_missing_before_after(df_train_fe, df_train_clean)

#### Save

In [ ]:
# save clean data to s3 for use in 04_model
for str_name, df_split in [('df_train_clean', df_train_clean), ('df_valid_clean', df_valid_clean), ('df_test_clean', df_test_clean)]:
    str_s3_output = f's3://{str_bucket}/{str_step}/{str_name}.parquet'
    df_split.to_parquet(str_s3_output, index=False)
    print(f'{str_name} saved to {str_s3_output}')